In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from adjustText import adjust_text
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"]
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42

In [ ]:
## paper data
df = pd.read_csv("results/fig3_labels_va_results_paper.csv")
cluster_result = "results/fig3_cluster_results_paper.csv"
fig_name = "results_fig/fig3_paper.png"

## results from a rerun using Japanese prompts
#df = pd.read_csv("results/fig3_labels_va_results_test_jp.csv")
#cluster_result = "results/fig3_cluster_results_test_jp.csv"
#fig_name = "results_fig/fig3_test_jp.png"

## results from a rerun using English prompts
#df = pd.read_csv("results/fig3_labels_va_results_test_en.csv")
#cluster_result = "results/fig3_cluster_results_test_en.csv"
#fig_name = "results_fig/fig3_test_en.png"

In [ ]:
# clustering

avg_df = (
    df.groupby(["ID", "emotion"], as_index=False)[["valence", "arousal"]]
    .mean()
)
X2 = avg_df[["valence", "arousal"]].to_numpy()


# KMeans
results = []
n = len(X2)

for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=0, n_init=20)
    cluster_labels = km.fit_predict(X2)
    score = silhouette_score(X2, cluster_labels)
    results.append((k, score))
    print(f"k={k}, silhouette={score:.4f}")

best_k, best_score = max(results, key=lambda x: x[1])
print("best_k =", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=0, n_init=20)
avg_df["cluster"] = kmeans.fit_predict(X2)

# reorder the clusters for easier interpretation (only for paper data)
if cluster_result == "results/fig3_cluster_results_paper.csv":
    mapping = {2: 0, 0: 1, 1: 2, 3: 3, 4: 4}
    avg_df["cluster"] = avg_df["cluster"].map(mapping)

# save results
cluster_df = avg_df[
    ["cluster", "ID", "valence", "arousal"]
].copy()

cluster_df["cluster"] = cluster_df["cluster"] + 1
cluster_df = cluster_df.sort_values(["cluster", "ID"]).reset_index(drop=True)

cluster_df.to_csv(cluster_result, index=False, encoding="utf-8-sig")
print(f"saved: {cluster_result}")

In [ ]:
# visualize

plt.figure(figsize=(10, 8))

texts = []

for c in sorted(avg_df["cluster"].unique()):
    sub = avg_df[avg_df["cluster"] == c]
    plt.scatter(sub["valence"], sub["arousal"], alpha=0.8, label=f"Cluster {c+1}")

for _, row in avg_df.iterrows():
    texts.append(
        plt.text(
            row["valence"],
            row["arousal"],
            row["ID"],
            fontsize=14,
            ha="center"
        )
    )

# adjust labels to avoid overlap.
adjust_text(
    texts,
    max_move=2,
)

plt.xlim(-1, 1)
plt.ylim(-1, 1)

plt.axhline(0, color="gray", linewidth=1)
plt.axvline(0, color="gray", linewidth=1)

ticks = np.arange(-1.0, 1.01, 0.25)
plt.xticks(ticks, fontsize=14)
plt.yticks(ticks, fontsize=14)

plt.grid(True, linestyle="--", linewidth=0.7, alpha=0.5)
plt.xlabel("Valence", fontsize=16)
plt.ylabel("Arousal", fontsize=16)
plt.legend(fontsize=16)
plt.tight_layout()
plt.savefig(fig_name, dpi=300, bbox_inches="tight")
plt.show()